##Datalab Semester 2

In [4]:
import os
import sqlite3
import pandas as pd

# 1. Zoek de map op waar dit notebook-bestand staat
map_van_notebook = os.path.dirname(os.path.abspath('__file__'))

# 2. Maak het volledige pad naar de database
db_pad = os.path.join(map_van_notebook, 'database.sqlite')
conn = sqlite3.connect(db_pad)

In [5]:
tabellen = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
tabellen


,name
0,sqlite_sequence
1,Player_Attributes
2,Player
3,Match
4,League
5,Country
6,Team
7,Team_Attributes


In [6]:
query = "SELECT * FROM Team"

df_teams = pd.read_sql_query(query, conn)
df_teams.head()

,id,team_api_id,team_fifa_api_id,team_long_name,team_short_name
0,1,9987,673.0,KRC Genk,GEN
1,2,9993,675.0,Beerschot AC,BAC
2,3,10000,15005.0,SV Zulte-Waregem,ZUL
3,4,9994,2007.0,Sporting Lokeren,LOK
4,5,9984,1750.0,KSV Cercle Brugge,CEB


In [7]:
# Automatisch het juiste pad vinden, ongeacht waar je bent op je Mac
current_dir = os.path.dirname(os.path.abspath('__file__'))
db_path = os.path.join(current_dir, 'database.sqlite')

print(f"Ik zoek de database op: {db_path}")

conn = sqlite3.connect(db_path)
query = "SELECT * FROM Team WHERE team_long_name LIKE '%Barcelona%'"

df = pd.read_sql_query(query, conn)
df.head()

Ik zoek de database op: c:\Users\sasha\Documents\GitHub\Datalab_semester2_Groep1\notebook\database.sqlite


,id,team_api_id,team_fifa_api_id,team_long_name,team_short_name
0,43042,8634,241,FC Barcelona,BAR


Ranglijst opstellen

In [ ]:

def bepaal_match_punten(row):
    """
    Berekent de punten voor de thuis- en uitploeg op basis van de wedstrijdscore.
    
    Args:
        row (pd.Series): Een rij uit de Match dataframe met 'home_team_goal' en 'away_team_goal'.
        
    Returns:
        pd.Series: De behaalde punten voor [home_points, away_points].
    """
    if row['home_team_goal'] > row['away_team_goal']:
        return pd.Series([3, 0], index=['home_points', 'away_points'])
    elif row['home_team_goal'] < row['away_team_goal']:
        return pd.Series([0, 3], index=['home_points', 'away_points'])
    else:
        return pd.Series([1, 1], index=['home_points', 'away_points'])

def genereer_competitie_ranglijst(connection, seizoen):
    """
    Haalt wedstrijddata op uit de database en genereert een gesorteerde ranglijst.
    
    Args:
        connection (sqlite3.Connection): De actieve database verbinding.
        seizoen (str): Het gewenste seizoen (bijv. '2015/2016').
        
    Returns:
        pd.DataFrame: Een dataframe met de teamnamen en hun totale punten, gesorteerd van hoog naar laag.
    """
    # 1. Data ophalen
    query = f"SELECT home_team_api_id, away_team_api_id, home_team_goal, away_team_goal FROM Match WHERE season = '{seizoen}'"
    df_matches = pd.read_sql_query(query, connection)
    
    # 2. Punten berekenen met de hulpfunctie
    df_matches[['home_points', 'away_points']] = df_matches.apply(bepaal_match_punten, axis=1)
    
    # 3. Groeperen en totalen berekenen
    home_stats = df_matches.groupby('home_team_api_id')['home_points'].sum().reset_index()
    away_stats = df_matches.groupby('away_team_api_id')['away_points'].sum().reset_index()
    
    home_stats.columns = ['team_api_id', 'points']
    away_stats.columns = ['team_api_id', 'points']
    
    ranglijst = pd.concat([home_stats, away_stats]).groupby('team_api_id').sum().reset_index()
    
    # 4. Teamnamen toevoegen
    df_teams_names = pd.read_sql_query("SELECT team_api_id, team_long_name FROM Team", connection)
    ranglijst = ranglijst.merge(df_teams_names, on='team_api_id')
    
    return ranglijst.sort_values(by='points', ascending=False).reset_index(drop=True)


df_2015_2016 = genereer_competitie_ranglijst(conn, '2015/2016')

# Resultaat bekijken
display(df_2015_2016[['team_long_name', 'points']].head(10))

,team_long_name,points
0,Paris Saint-Germain,96
1,FC Barcelona,91
2,Juventus,91
3,Real Madrid CF,90
4,SL Benfica,88
5,Atlético Madrid,88
6,FC Bayern Munich,88
7,Sporting CP,86
8,Celtic,86
9,PSV,84


: 